### Langchain Agents

In [ ]:
import os
from dotenv import load_dotenv
from pydantic import SecretStr

from langchain_openai import ChatOpenAI

load_dotenv(override=True)

OPENAI_BASE_URL: str = os.getenv("OPENAI_BASE_URL", "")
OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
LLM_MODEL_NAME: str = os.getenv("OPENAI_MODEL_NAME", "")

llm = ChatOpenAI(
    base_url=OPENAI_BASE_URL,
    api_key=SecretStr(OPENAI_API_KEY),
    model=LLM_MODEL_NAME,
    temperature=0,
    max_completion_tokens=1024,
    reasoning_effort="none",
)

In [ ]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get the weather for a city"""
    return f"The weather in {city} is sunny."


agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)

agent

In [ ]:
agent.invoke(input={"messages": "What is the weather in Kolkata?"})  # type: ignore

In [ ]:
### Batch processing

llm.batch(
    [
        "What is the capital of France?",
        "What is the value of factorial 3?",
        "Do you think AI agents are the future in computer science?",
    ],
    config={
        "max_concurrency": 5,
    },
)

### Structured Output

In [ ]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movies rating out of 10")

In [ ]:
model = llm.with_structured_output(schema=Movie)

response = model.invoke(input="Provide details about the movie Inception")
response

In [ ]:
model = llm.with_structured_output(schema=Movie, include_raw=True)

response = model.invoke("Provide details about the movie Inception")
response

In [ ]:
class Actor(BaseModel):
    name: str = Field(description="The name of the actor")
    role: str = Field(description="The role of the actor in this movie")


class MovieDetails(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    genres: list[str] = Field(description="The list of genre this movie belongs too")
    cast: list[Actor] = Field(description="The list of casts present in this movie")
    budget: float | None = Field(
        default=None, description="The movie budget in millions USD"
    )

In [ ]:
model = llm.with_structured_output(MovieDetails)

response = model.invoke(input="Provide details about the movie Inception")
response

In [ ]:
print(response.genres)  # type: ignore
print(response.budget)  # type: ignore
for cast in response.cast:  # type: ignore
    print(cast.name + " -> " + cast.role)

In [ ]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [ ]:
model = llm.with_structured_output(schema=MovieDict)
response = model.invoke(input="Please provided the details of the movie Avengers")
response

In [ ]:
llm.profile

### Agent with structure response

In [ ]:
class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    phone: str = Field(description="The phone number of the person")

In [ ]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant.",
    response_format=ContactInfo,
)

agent

In [ ]:
response = agent.invoke(
    {
        "messages": "Extract the contact information from text: Tom Holland, +1234567, tom@email.com"
    }  # type: ignore
)
response

In [ ]:
response["structured_response"]

### Middlewares

##### Summarization Middleware

In [ ]:
import os
from dotenv import load_dotenv
from pydantic import SecretStr

from langchain_openai import ChatOpenAI

load_dotenv(override=True)

OPENAI_BASE_URL: str = os.getenv("OPENAI_BASE_URL", "")
OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
LLM_MODEL_NAME: str = os.getenv("OPENAI_MODEL_NAME", "")

llm = ChatOpenAI(
    base_url=OPENAI_BASE_URL,
    api_key=SecretStr(OPENAI_API_KEY),
    model=LLM_MODEL_NAME,
    temperature=0,
    max_completion_tokens=1024,
    reasoning_effort="none",
)
llm

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

# Message based summarization
agent = create_agent(
    model=llm,
    system_prompt="You are a smart, responsible and intelligent assistant agent.",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)

agent

In [ ]:
# Run with thread id
config = {"configurable": {"thread_id": "test-123"}}

In [ ]:
# Test questions
questions: list[str] = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 3!?",
    "What is 3n2?",
    "What is 2^2?",
    "What is 10%3?",
]

for question in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config,  # type: ignore
    )
    print(f"Messages: {response}")
    print(f"Count: {len(response['messages'])}")

In [ ]:
from langchain.tools import tool


@tool
def search_hotels(city: str) -> str:
    """Search hotels in the city."""

    return f"""Hotel in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free WiFi"""


# Token based summarization
agent = create_agent(
    model=llm,
    system_prompt="You are a smart, responsible and intelligent assistant agent.",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 512),
            keep=("tokens", 128),
        )
    ],
)

config = {"configurable": {"thread_id": "test-123"}}

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]
for city in cities:
    response = agent.invoke(
        input={"messages": [HumanMessage(content=f"Find hotels in city: {city}")]},
        config=config,  # type: ignore
    )

    print(f"{city}: {len(response['messages'])} messages")
    print(f"{response['messages']}")

### Human in the Loop

In [ ]:
import os
from dotenv import load_dotenv
from pydantic import SecretStr

from langchain_openai import ChatOpenAI

load_dotenv(override=True)

OPENAI_BASE_URL: str = os.getenv("OPENAI_BASE_URL", "")
OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
LLM_MODEL_NAME: str = os.getenv("OPENAI_MODEL_NAME", "")

llm = ChatOpenAI(
    base_url=OPENAI_BASE_URL,
    api_key=SecretStr(OPENAI_API_KEY),
    model=LLM_MODEL_NAME,
    temperature=0,
    max_completion_tokens=1024,
    reasoning_effort="none",
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [ ]:
agent = create_agent(
    model=llm,
    system_prompt="You are a smart and intelligent assistant.",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
                "read_email_tool": False,
            }
        )
    ],
)

##### Approving the tool call

In [ ]:
config = {"configurable": {"thread_id": "test-approve"}}

result = agent.invoke(
    input={
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello from Mike' and body 'How are you doing?'"
            )
        ]
    },
    config=config,  # type: ignore
)

result

In [ ]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ paused! Approving...")

    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,  # type: ignore
    )

    print(f"✅ Result: {result['messages'][-1].content}")

result

##### Rejecting the tool call

In [ ]:
config = {"configurable": {"thread_id": "test-reject"}}

result = agent.invoke(
    input={
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello from Mike' and body 'How are you doing?'"
            )
        ]
    },
    config=config,  # type: ignore
)

result

In [ ]:
if "__interrupt__" in result:
    print("⏸️ paused! Rejecting...")

    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}),
        config=config,  # type: ignore
    )

    print(f"✅ Result: {result['messages'][-1].content}")

result

##### Editing the tool call response

In [ ]:
config = {"configurable": {"thread_id": "test-edit"}}

result = agent.invoke(
    input={
        "messages": [
            HumanMessage(
                content="Send email to wrong@test.com with subject 'Hello from Agent' and body 'How are you doing?'"
            )
        ]
    },
    config=config,  # type: ignore
)

result

In [ ]:
if "__interrupt__" in result:
    print("⏸️ paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct@test.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by Human before sending",
                            },
                        },
                    }
                ]
            }
        ),
        config=config,  # type: ignore
    )

    print(f"✅ Result: {result['messages'][-1].content}")

result